# 5 · The E-step: `_expected_counts`

**Trusts:** `_unnormalized` and `_joint` (notebooks 2–3), `_complete_counts` (notebook 4).

Counting needs complete rows. EM's idea: a row with CO missing still counts, **fractionally**. If P(CO | row) = (0.3, 0.7), the row adds 0.3 to the CO = 0 cell and 0.7 to the CO = 1 cell. Every row contributes a total weight of exactly 1 to each family, spread over the hidden possibilities.

We build it as: **complete row → one hidden node → two hidden nodes in one family → general → every row.**

In [ ]:
%load_ext autoreload
%autoreload 2

import itertools

import numpy as np
import pandas as pd
from toy import (
    CARDS,
    CO,
    DATA,
    EVIDENCE,
    NO2,
    S1,
    S4,
    STRUCTURE,
    TABLES,
    H,
    T,
    cpts,
    network,
    same,
    toy_bn,
)

from cse250a.modules.core import CPT

net = network()
bn = toy_bn()   # DiscreteBN with the hand-picked CPTs, not fitted

In [ ]:
def empty():
    return {n: np.zeros(net.shape(n)) for n in net.structure}

## Step 1: a complete row adds 1 to one cell per family

Nothing hidden means nothing is uncertain: each family's observed values pick one cell, and it gets +1. That's exactly the complete-case count of that single row.

In [ ]:
row = DATA.iloc[0]
step1 = empty()
for node in net.structure:
    index = tuple(int(row[c]) for c in net.family(node))
    step1[node][index] += 1

same("complete row = its complete count", step1, bn._complete_counts(DATA.iloc[[0]]))
same("complete row = bn E-step", step1, bn._expected_counts(DATA.iloc[[0]])[0])

## Step 2: one hidden node

Row 2: CO missing. P(CO | row) comes from `_joint` (trusted). Now go family by family:

| family | CO in it? | what to add |
|---|---|---|
| CO: (hour, CO) | yes, as the node | the whole vector P(CO \| row) into `counts[CO][hour, :]` |
| NO2: (hour, NO2) | no | +1 at the observed cell |
| S1: (CO, T, S1) | yes, as a parent | P(CO \| row) into `counts[S1][:, t, s1]` |
| S4: (NO2, CO, S4) | yes, as a parent | P(CO \| row) into `counts[S4][no2, :, s4]` |

`:` (a slice) marks the hidden axis; the observed columns pick a single index.

In [ ]:
row = DATA.iloc[2]
h, t, no2, s1, s4 = (int(row[c]) for c in (H, T, NO2, S1, S4))
hidden, joint = bn._joint(row)
p = joint                       # one hidden node, so the joint IS P(CO | row)
print("hidden:", [str(x) for x in hidden], " P(CO | row) =", p.round(4))

step2 = empty()
step2[CO][h, :] += p
step2[NO2][h, no2] += 1
step2[S1][:, t, s1] += p
step2[S4][no2, :, s4] += p

same("one hidden node", step2, bn._expected_counts(DATA.iloc[[2]])[0])

## Step 3: two hidden nodes, and the axis-order trap

Row 4: CO **and** NO2 missing. `_joint` gives a 2×2 table with axes in **`hidden` order**, `(CO, NO2)`, because `hidden` follows `structure` order.

Each family needs the joint of **its own** hidden members, so we sum away the rest. For CO's family that's summing over NO2. S4's family is `(NO2, CO, S4)` with both gases hidden, so it needs P(NO2, CO | row) with axes **(NO2, CO)**, the other way round from the joint. Forget to transpose and the numbers land in the wrong cells:

In [ ]:
row = DATA.iloc[4]
h, t, s1, s4 = (int(row[c]) for c in (H, T, S1, S4))
hidden, joint = bn._joint(row)
print("joint axes:", [str(x) for x in hidden])

p_co, p_no2 = joint.sum(axis=1), joint.sum(axis=0)

def build(s4_block):
    c = empty()
    c[CO][h, :] += p_co
    c[NO2][h, :] += p_no2
    c[S1][:, t, s1] += p_co
    c[S4][:, :, s4] += s4_block       # family axes: (NO2, CO, S4)
    return c

theirs = bn._expected_counts(DATA.iloc[[4]])[0]
same("without transpose (wrong on purpose: expect ❌)", build(joint)[S4], theirs[S4])
same("with joint.T (right)      ", build(joint.T)[S4], theirs[S4])
same("all families", build(joint.T), theirs)

## Step 4: the general rule, for any row and any family

For each family:

1. `fam_hidden`: the family members that are hidden, **in family order**
2. `axes`: where each of them sits in the joint (`hidden.index`)
3. sum the joint over every *other* axis
4. the result's axes are now in **sorted** axis order, so transpose them into family order (step 3's trap, handled in general)
5. index the counts with a slice for each hidden member and the observed bin for the rest, then add

In [ ]:
def add_row(counts, row):
    hidden, joint = bn._joint(row)
    for node in net.structure:
        family = net.family(node)
        fam_hidden = [c for c in family if c in hidden]                      # 1
        axes = [hidden.index(c) for c in fam_hidden]                         # 2
        others = tuple(a for a in range(joint.ndim) if a not in axes)
        small = joint.sum(axis=others)                                       # 3
        order = sorted(axes)
        small = np.transpose(small, [order.index(a) for a in axes])          # 4
        index = tuple(slice(None) if c in hidden else int(row[c]) for c in family)
        counts[node][index] += small                                         # 5

for i in range(len(DATA)):
    c = empty()
    add_row(c, DATA.iloc[i])
    same(f"row {i}", c, bn._expected_counts(DATA.iloc[[i]])[0])

**Step 4's transpose, on its own.** Say the joint's axes are `(CO, NO2, S1)` and a family wants `(NO2, CO)`. Then `axes = [1, 0]`; after summing S1 away, the result's axes are in sorted order `[0, 1]` = (CO, NO2); `[order.index(a) for a in axes]` = `[1, 0]` swaps them into (NO2, CO). When the family order already matches, it's the identity, so it's always safe to apply.

## Step 5: every row, plus the log-likelihood

Sum `add_row` over all rows. The method also returns Σ log(table.sum()), which is the log-likelihood from notebook 3, computed in the same pass so EM can track it.

In [ ]:
mine = empty()
for _, row in DATA.iterrows():
    add_row(mine, row)

counts, loglik = bn._expected_counts(DATA)
same("all rows", mine, counts)
same("its loglik = log_likelihood", loglik, bn.log_likelihood(DATA))

**Sanity check:** every row adds a total of exactly 1 to every family, observed or not. So each family's counts sum to the number of rows. Compare `_complete_counts`, which only uses some of them:

In [ ]:
complete = bn._complete_counts(DATA)
for node in net.nodes:
    print(f"{node!s:>14}: expected counts sum to {counts[node].sum():.6f}   complete-case: {int(complete[node].sum())}")